# Patch-Based Binary Segmentation

> Patch-based training for 3D medical image segmentation using fastMONAI's `MedPatchDataLoaders` with **lazy loading** - volumes are loaded on-demand, keeping memory usage constant regardless of dataset size.

This tutorial demonstrates when and how to use patch-based training:

- **Large volumes**: When full images don't fit in GPU memory
- **Memory constraints**: Constant ~150 MB memory usage regardless of dataset size
- **Class imbalance**: Foreground-weighted sampling for better learning on small structures

We use the same Heart MRI dataset from the [binary segmentation tutorial](11d_tutorial_binary_segmentation.ipynb) for direct comparison.

[![Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MMIV-ML/fastMONAI/blob/main/nbs/12a_tutorial_patch_training.ipynb)

In [ ]:
#| hide
#Install `fastMONAI` if notebook is running on Google Colab
try:
    import google.colab
    %pip install fastMONAI
    from fastMONAI.utils import print_colab_gpu_info
    print_colab_gpu_info()
except:
    print('Running locally')

In [ ]:
from fastMONAI.vision_all import *

from monai.apps import DecathlonDataset
from sklearn.model_selection import train_test_split

### Download external data

We use the MONAI function `DecathlonDataset` to download the Heart MRI dataset from the Medical Segmentation Decathlon challenge.

In [ ]:
path = Path('../data')
path.mkdir(exist_ok=True)

In [ ]:
task = "Task02_Heart"
training_data = DecathlonDataset(root_dir=path, task=task, section="training", 
    download=True, cache_num=0, num_workers=3)

In [ ]:
df = pd.DataFrame(training_data.data)
df.shape

Split the labeled data into training and test sets.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_df.shape, test_df.shape

### Analyze training data

Use `MedDataset` to analyze the dataset and get preprocessing recommendations.

In [ ]:
med_dataset = MedDataset(img_list=train_df.label.tolist(), dtype=MedMask, max_workers=12)

In [ ]:
med_dataset.df.head()

In [ ]:
data_info_df = med_dataset.summary()

In [ ]:
suggestion = med_dataset.get_suggestion()
target_spacing, apply_reorder = suggestion['target_spacing'], suggestion['apply_reorder']
target_spacing, apply_reorder

In [ ]:
stats = med_dataset.get_size_statistics(target_spacing=target_spacing)
print(f"Image sizes (after resampling to {target_spacing}):")
print(f"  Min:    {stats['min']}")
print(f"  Median: {stats['median']}")
print(f"  Max:    {stats['max']}")

suggested_size = suggest_patch_size(med_dataset, target_spacing=target_spacing)
print(f"\nSuggested patch size: {suggested_size}")


### Configure patch-based training

`PatchConfig` centralizes all patch-related parameters:

- **patch_size**: Size of extracted patches `[x, y, z]` - should be divisible by 16 for UNet compatibility
- **samples_per_volume**: Number of patches extracted per volume per epoch
- **sampler_type**: `'uniform'` (random) or `'label'` (foreground-weighted)
- **label_probabilities**: For `'label'` sampler, probability of sampling each class
- **queue_length**: Number of patches to keep in memory buffer
- **patch_overlap**: Overlap for inference (float 0-1 for fraction, or int for pixels)
- **aggregation_mode**: How to combine overlapping patches (`'hann'` for smooth boundaries)
- **padding_mode**: Padding mode when image < patch_size (0 = zero padding, nnU-Net standard)

In [ ]:
patch_config = PatchConfig(
    patch_size=[128, 128, 64],
    samples_per_volume=8,
    sampler_type='label',
    label_probabilities={0: 0.2, 1: 0.8},
    patch_overlap=0.5,
    keep_largest_component=True,
    target_spacing=target_spacing,
    apply_reorder=apply_reorder
)

print(f"Patch config: {patch_config}")

> **Alternative**: Use `PatchConfig.from_dataset(med_dataset)` to auto-configure patch size based on dataset analysis.

### Define transforms

Patch-based training uses **two stages** of transforms:

1. **pre_patch_tfms**: Applied to full volumes before patch extraction (e.g., normalization)
2. **patch_tfms**: Applied to extracted patches during training (augmentations)

> **Critical**: `pre_patch_tfms` must match between training and inference for consistent preprocessing.

In [ ]:
# Pre-patch transforms (applied to full volumes by Queue workers)
pre_patch_tfms = [ZNormalization()]

# Patch augmentations (applied to training patches only)
patch_tfms = [
    RandomAffine(scales=(0.95, 1.3), degrees=15, translation=5, p=0.5),
    RandomGamma(log_gamma=(-0.3, 0.3), p=0.5),
    RandomBiasField(coefficients=0.5, p=0.3),
    RandomBlur(std=(0.0, 0.8), p=0.2),
    RandomNoise(std=0.1, p=0.2),
    RandomFlip(p=0.5),
]

### Create patch-based DataLoaders

`MedPatchDataLoaders` uses **lazy loading**:
- Only file paths are stored at creation time (~0 MB)
- Volumes are loaded on-demand by Queue workers
- Memory usage stays constant (~150 MB) regardless of dataset size

In [ ]:
bs = 4

dls = MedPatchDataLoaders.from_df(
    df=train_df,
    img_col='image',
    mask_col='label',
    valid_pct=0.1,
    patch_config=patch_config,
    pre_patch_tfms=pre_patch_tfms,
    patch_tfms=patch_tfms,
    bs=bs,
    seed=42
)

print(f"Training subjects: {len(dls.train.subjects_dataset)}")
print(f"Validation subjects: {len(dls.valid.subjects_dataset)}")

In [ ]:
# Visualize a batch of patches
batch = next(iter(dls.train))
x, y = batch
print(f"Batch shape - Image: {x.shape}, Mask: {y.shape}")

### Create and train a 3D model

We use MONAI's UNet with:
- **out_channels=2**: Softmax output (background + foreground)
- **Instance normalization**: Common in medical imaging
- **DiceCELoss**: Combines Dice loss with Cross-Entropy for stable training

In [ ]:
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.losses import DiceCELoss

In [ ]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.INSTANCE
)

loss_func = CustomLoss(loss_func=DiceCELoss(
    to_onehot_y=True,
    softmax=True,
    include_background=True
))

We use `AccumulatedDice` metric which accumulates true positives, false positives, and false negatives across all validation batches before computing Dice. This is more statistically robust than averaging per-batch Dice scores (nnU-Net approach).

In [ ]:
learn = Learner(dls, model, loss_func=loss_func, metrics=[AccumulatedDice(n_classes=2)])

In [ ]:
learn.lr_find()

In [ ]:
lr = 1e-3

In [ ]:
save_best = SaveModelCallback(
    monitor='accumulated_dice',
    comp=np.greater,
    fname='best_heart_patch',
    every_epoch=False,
    with_opt=False
)

In [ ]:
import mlflow

task_name = "Task02_Heart_Patch"
mlflow.set_experiment(task_name)

mlflow_callback = ModelTrackingCallback(
    model_name=f"{task_name}_{model._get_name()}",
    loss_function=loss_func.loss_func._get_name(),
    item_tfms=pre_patch_tfms,
    size=patch_config.patch_size,
    target_spacing=target_spacing,
    apply_reorder=apply_reorder,
)

with mlflow.start_run(run_name="patch_training"):
    learn.fit_one_cycle(100, lr, cbs=[mlflow_callback, save_best])

In [ ]:
learn.recorder.plot_loss();

### Save configuration for inference

The following parameters **MUST be identical** between training and inference:
- `apply_reorder`: Whether to reorder to RAS+ orientation
- `target_spacing`: Target voxel spacing for resampling
- `pre_patch_tfms`: Preprocessing transforms (e.g., `ZNormalization`)

Save these to a pickle file for the inference notebook to load.

In [ ]:
store_patch_variables(
    pkl_fn='patch_config.pkl',
    patch_size=patch_config.patch_size,
    patch_overlap=patch_config.patch_overlap,
    aggregation_mode=patch_config.aggregation_mode,
    apply_reorder=apply_reorder,
    target_spacing=target_spacing,
    sampler_type=patch_config.sampler_type,
    label_probabilities=patch_config.label_probabilities,
    samples_per_volume=patch_config.samples_per_volume,
    queue_length=patch_config.queue_length,
    queue_num_workers=patch_config.queue_num_workers,
    keep_largest_component=patch_config.keep_largest_component
)

print("Configuration saved to patch_config.pkl")

In [ ]:
learn.save('heart-patch-weights')

### View experiment tracking

In [ ]:
mlflow_ui = MLflowUIManager()
mlflow_ui.start_ui()

### Summary

In this tutorial, we demonstrated patch-based training for 3D medical image segmentation:

1. **`PatchConfig`**: Centralized configuration for patch size, sampling, and inference parameters
2. **`MedPatchDataLoaders`**: Memory-efficient lazy loading with TorchIO Queue
3. **Two-stage transforms**: `pre_patch_tfms` (full volume) + `patch_tfms` (training augmentation)
4. **`AccumulatedDice`**: nnU-Net-style accumulated metric for reliable validation
5. **`store_patch_variables()`**: Save configuration for inference consistency

**Next step**: See [12b_tutorial_patch_inference.ipynb](12b_tutorial_patch_inference.ipynb) for sliding-window inference on test data.

In [ ]:
mlflow_ui.stop()